# Lab 5: Spark SQL, Window Functions & Performance Tuning
**Z5008 Big Data Lab | IIT Madras Zanzibar | Dr. Innocent Nyalala**
**Session 5**

## Learning Objectives
- **Part 1:** Master basic Spark SQL operations (Filtering, Aggregations, Data Manipulation, Joins, and Views).
- **Part 2:** Write advanced Spark SQL (CTEs, Lateral views, Array manipulation).
- **Part 3:** Apply window functions: rolling averages, LAG/LEAD, RANK, NTILE.
- **Part 4:** Understand partition sizing: `repartition` vs `coalesce`.
- **Part 5:** Create and query bucketed tables to avoid expensive network shuffles.
- **Part 6:** Enable and observe Adaptive Query Execution (AQE) optimizations.
- **Part 7:** Detect and fix data skew using the "Salting" technique.
- **Part 8:** Benchmark and compare execution times before and after optimizations.

In [ ]:
# ── SETUP CELL ── Run this first before anything else ──────────────────────────

import os
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

cwd = os.getcwd()
jar1_path = os.path.join(cwd, "hadoop-aws-3.3.4.jar")
jar2_path = os.path.join(cwd, "aws-java-sdk-bundle-1.12.262.jar")

print("Checking JAR files...")
if not os.path.exists(jar1_path):
    urllib.request.urlretrieve(
        "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
        jar1_path
    )
if not os.path.exists(jar2_path):
    urllib.request.urlretrieve(
        "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar",
        jar2_path
    )

jars_list = f"{jar1_path},{jar2_path}"
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    f"--jars {jars_list} "
    "--packages io.delta:delta-core_2.12:2.3.0 pyspark-shell"
)

print("Building Spark Session...")
spark = (
    SparkSession.builder
    .appName('Lab05-SparkSQL')
    .master('local[*]')
    .config("spark.hadoop.fs.s3a.endpoint",   "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123")
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled",  "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

ORDERS_PATH = 's3a://warehouse/lab04/orders_parquet/'

try:
    print("Loading existing data...")
    df_orders = spark.read.parquet(ORDERS_PATH)
except Exception:
    print("Data missing. Generating sample rows for Lab 05...")
    countries = ['Kenya', 'Tanzania', 'Nigeria', 'Uganda', 'Ghana', 'Ethiopia']
    categories = ['Electronics', 'Home', 'Gadgets', 'Furniture', 'Office', 'Clothing', 'Fashion']
    status_list = ['completed', 'pending', 'cancelled']
    
    df_orders = (
        spark.range(0, 100_000)
        .withColumn('order_id', F.concat(F.lit('ORD'), F.lpad(F.col('id'), 7, '0')))
        .withColumn('country', F.element_at(F.array([F.lit(c) for c in countries]), (F.rand() * len(countries) + 1).cast('int')))
        .withColumn('category', F.element_at(F.array([F.lit(c) for c in categories]), (F.rand() * len(categories) + 1).cast('int')))
        .withColumn('status', F.element_at(F.array([F.lit(s) for s in status_list]), (F.rand() * len(status_list) + 1).cast('int')))
        .withColumn('order_total', (F.rand() * 5000).cast('double'))
        .withColumn('order_date', F.from_unixtime(F.unix_timestamp(F.lit('2025-01-01')) + F.rand() * 86400 * 365).cast('timestamp'))
    )
    df_orders.write.mode('overwrite').parquet(ORDERS_PATH)
    print("Sample data generated and saved.")

df_orders.createOrReplaceTempView('orders')
print("\nStatus: SUCCESSFUL")
print(f"Records loaded into 'orders' view: {df_orders.count():,}")

## Part 1: Basic SQL Operations

In [ ]:
spark.sql("""
    SELECT order_id, country, ROUND(order_total, 2) AS total_amount 
    FROM orders 
    WHERE country = 'Kenya' AND order_total > 100 
    ORDER BY order_total DESC 
    LIMIT 5
""").show()

In [ ]:
spark.sql("""
    SELECT country, COUNT(order_id) AS total_orders, ROUND(AVG(order_total), 2) AS avg_spend 
    FROM orders 
    GROUP BY country 
    ORDER BY total_orders DESC
""").show()

## Part 5: Bucketing and Performance

In [ ]:
spark.sql("DROP TABLE IF EXISTS orders_bucketed")
df_orders.write.bucketBy(8, 'country').sortBy('country').mode('overwrite').saveAsTable('orders_bucketed')
print("Bucketed table created.")

In [ ]:
# Fixed Syntax Error: Removed backslash followed by comment
print('=== Plan WITHOUT bucketing ===')
df_orders.groupBy('country').count().explain(mode='simple')

## Part 7: Data Skew and Salting

In [ ]:
from pyspark.sql import functions as F
skewed = spark.range(0, 1_000_000).withColumn('country', F.when(F.col('id') % 10 < 9, F.lit('Kenya')).otherwise(F.lit('Tanzania')))
skewed.groupBy('country').count().show()